In [2]:
from pyspark.sql import SparkSession
import getpass

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("Colab_Spark_RDD") \
    .getOrCreate()

# Lấy SparkContext từ SparkSession để dùng RDD
sc = spark.sparkContext

print("Spark version:", sc.version)

Spark version: 4.0.1


# **Bài 4: Phân Tích Đánh Giá Theo Nhóm Tuổi**

In [3]:
movies_rdd = sc.textFile("data/movies.txt")
ratings_rdd = sc.textFile("data/ratings_*.txt")
user_rdd = sc.textFile("data/users.txt")
occupation_rdd = sc.textFile("data/occupations.txt")

In [4]:
movies_rdd.take(5)

['1001,The Godfather (1972),Crime|Drama',
 '1002,The Shawshank Redemption (1994),Drama',
 "1003,Schindler's List (1993),Biography|Drama|History",
 '1004,Raging Bull (1980),Biography|Drama|Sport',
 '1005,Casablanca (1942),Drama|Romance|War']

In [5]:
ratings_rdd.take(5)

['7,1020,4.5,1577836800',
 '23,1015,3.5,1577923200',
 '45,1030,4.0,1578009600',
 '12,1047,3.0,1578096000',
 '38,1012,4.5,1578182400']

In [6]:
user_rdd.take(5)

['1,M,28,3,12345',
 '2,F,35,7,23456',
 '3,M,42,2,34567',
 '4,F,19,10,45678',
 '5,M,31,1,56789']

In [9]:
#Phan loai nhom tuoi
def age_group(age):
    if age < 18:
        return "0-18"
    elif 18 <= age < 35:
        return "18-35"
    elif 35 <= age <= 50:
        return "35-50"
    else:
        return "50+"
    
#Mapper user lay id va nhom tuoi
user_mapped = user_rdd.map(lambda x: (x.split(",")[0], age_group(int(x.split(",")[2]))))
user_mapped.take(5)

[('1', '18-35'),
 ('2', '35-50'),
 ('3', '35-50'),
 ('4', '18-35'),
 ('5', '18-35')]

In [10]:
#Mapper lay movie
movie_mapped = movies_rdd.map(lambda x: (x.split(",")[0], x.split(",")[1]))
movie_mapped.take(5)

[('1001', 'The Godfather (1972)'),
 ('1002', 'The Shawshank Redemption (1994)'),
 ('1003', "Schindler's List (1993)"),
 ('1004', 'Raging Bull (1980)'),
 ('1005', 'Casablanca (1942)')]

In [13]:
#Movie lay rating
rating_mapped = ratings_rdd.map(lambda x: (x.split(",")[0], (x.split(",")[1], x.split(",")[2])))
rating_mapped.take(5)

[('7', ('1020', '4.5')),
 ('23', ('1015', '3.5')),
 ('45', ('1030', '4.0')),
 ('12', ('1047', '3.0')),
 ('38', ('1012', '4.5'))]

In [14]:
#Join rating va user theo userID
rating_user_joined = rating_mapped.join(user_mapped)
rating_user_joined.take(5)

[('12', (('1047', '3.0'), '35-50')),
 ('12', (('1012', '3.5'), '35-50')),
 ('12', (('1040', '4.0'), '35-50')),
 ('12', (('1013', '4.5'), '35-50')),
 ('50', (('1025', '4.5'), '35-50'))]

In [15]:
#Map lai lay movieId lam key
rating_user_joined = rating_user_joined.map(lambda x: (x[1][0][0], (x[1][0][1], x[1][1])))
rating_user_joined.take(5)

[('1047', ('3.0', '35-50')),
 ('1012', ('3.5', '35-50')),
 ('1040', ('4.0', '35-50')),
 ('1013', ('4.5', '35-50')),
 ('1025', ('4.5', '35-50'))]

In [26]:
#join movie va tinh avg rating theo nhom tuoi
movie_rating_agegroup = movie_mapped.join(rating_user_joined)

movie_rating_final = movie_rating_agegroup.map(lambda x: ((x[1][0], x[1][1][1]), (float(x[1][1][0]), 1)))

totalRatings = movie_rating_final.reduceByKey(lambda a, b: (a[0]+b[0],a[1]+b[1]))
avgRatings = totalRatings.mapValues(lambda x: x[0]/x[1])

#Group lai theo movieId
final_result = avgRatings.map(lambda x: (x[0][0], (x[0][1], x[1]))).groupByKey().mapValues(lambda x: sorted(list(x)))
final_result.take(5)



[('Mad Max: Fury Road (2015)',
  [('18-35', 3.3636363636363638), ('35-50', 3.642857142857143)]),
 ('The Terminator (1984)',
  [('18-35', 4.166666666666667), ('35-50', 4.05), ('50+', 3.75)]),
 ('The Godfather: Part II (1974)',
  [('18-35', 3.7777777777777777), ('35-50', 4.25)]),
 ('The Silence of the Lambs (1991)', [('18-35', 3.0), ('35-50', 3.25)]),
 ('The Lord of the Rings: The Return of the King (2003)',
  [('18-35', 3.8333333333333335), ('35-50', 3.8125)])]

In [27]:
#Format output
def format_output(record):
    movieTitle = record[0]
    ratings = dict(record[1])
    
    if "0-18" in ratings:
        rating_0_18 = f"{ratings['0-18']:.2f}"
    else:
        rating_0_18 = "N/A"
    if "18-35" in ratings:
        rating_18_35 = f"{ratings['18-35']:.2f}"
    else:
        rating_18_35 = "N/A"
    if "35-50" in ratings:
        rating_35_50 = f"{ratings['35-50']:.2f}"
    else:
        rating_35_50 = "N/A"
    if "50+" in ratings:
        rating_50_plus = f"{ratings['50+']:.2f}"
    else:
        rating_50_plus = "N/A"
    return f"{movieTitle} - [0-18: {rating_0_18}, 18-35: {rating_18_35}, 35-50: {rating_35_50}, 50+: {rating_50_plus}]"

formatted_result = final_result.map(format_output)
formatted_result.collect()

['Mad Max: Fury Road (2015) - [0-18: N/A, 18-35: 3.36, 35-50: 3.64, 50+: N/A]',
 'The Terminator (1984) - [0-18: N/A, 18-35: 4.17, 35-50: 4.05, 50+: 3.75]',
 'The Godfather: Part II (1974) - [0-18: N/A, 18-35: 3.78, 35-50: 4.25, 50+: N/A]',
 'The Silence of the Lambs (1991) - [0-18: N/A, 18-35: 3.00, 35-50: 3.25, 50+: N/A]',
 'The Lord of the Rings: The Return of the King (2003) - [0-18: N/A, 18-35: 3.83, 35-50: 3.81, 50+: N/A]',
 'No Country for Old Men (2007) - [0-18: N/A, 18-35: 3.79, 35-50: 3.94, 50+: 4.00]',
 'Lawrence of Arabia (1962) - [0-18: N/A, 18-35: 3.60, 35-50: 3.29, 50+: 4.50]',
 'The Social Network (2010) - [0-18: N/A, 18-35: 4.00, 35-50: 3.67, 50+: N/A]',
 'Psycho (1960) - [0-18: N/A, 18-35: 4.50, 35-50: 3.50, 50+: N/A]',
 'Gladiator (2000) - [0-18: N/A, 18-35: 3.43, 35-50: 3.78, 50+: 3.50]',
 'Sunset Boulevard (1950) - [0-18: N/A, 18-35: 4.17, 35-50: 4.50, 50+: N/A]',
 'The Lord of the Rings: The Fellowship of the Ring (2001) - [0-18: N/A, 18-35: 4.00, 35-50: 3.83, 50+

----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 50838)
Traceback (most recent call last):
  File "c:\Users\lam_edora\AppData\Local\Programs\Python\Python310\lib\socketserver.py", line 316, in _handle_request_noblock
    self.process_request(request, client_address)
  File "c:\Users\lam_edora\AppData\Local\Programs\Python\Python310\lib\socketserver.py", line 347, in process_request
    self.finish_request(request, client_address)
  File "c:\Users\lam_edora\AppData\Local\Programs\Python\Python310\lib\socketserver.py", line 360, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "c:\Users\lam_edora\AppData\Local\Programs\Python\Python310\lib\socketserver.py", line 747, in __init__
    self.handle()
  File "c:\Users\lam_edora\AppData\Local\Programs\Python\Python310\lib\site-packages\pyspark\accumulators.py", line 299, in handle
    poll(accum_updates)
  File "c:\Users\lam_edora\AppData\Local\Progr